In [ ]:
import google.generativeai as genai
import os
import json
from google.api_core import retry

class AIStudioCRNMutator:
    def __init__(self, api_key: str):
        """
        Args:
            api_key: Your string key starting with 'AIza...' from Google AI Studio.
        """
        # 1. Configure the AI Studio SDK
        genai.configure(api_key=api_key)
        
        # 2. Define System Instructions
        # In AI Studio SDK, we pass this directly to the model constructor.
        self.system_instruction = """
        You are an expert Synthetic Biologist specialized in Mass Action Kinetics.
        You act as a mutation operator in an evolutionary algorithm.
        You ALWAYS output raw JSON. No markdown formatting.
        """

        # 3. Configure the Model
        # We use the alias 'gemini-1.5-flash' which points to the latest stable version.
        self.model = genai.GenerativeModel(
            model_name='gemini-1.5-flash',
            system_instruction=self.system_instruction,
            generation_config={
                "temperature": 0.8,
                "response_mime_type": "application/json"
            }
        )

    def propose_mutation(self, current_crn_str: str, task_description: str):
        
        # 4. Define the Prompt (Cleaner, since system prompt is handled above)
        prompt = f"""
        TASK: {task_description}

        CURRENT CRN TOPOLOGY:
        {current_crn_str}

        OBJECTIVE:
        The current network is getting stuck in a local optimum.
        Suggest 2 structural changes (additions) to improve robustness.
        
        OUTPUT FORMAT (JSON ONLY):
        {{
            "reasoning": "One sentence explanation",
            "new_reactions": ["A + B -> C; [MAK(1.0)]", ...]
        }}
        """

        try:
            # 5. Call the API
            # Note: We rely on the generation_config set in __init__, 
            # but we can override it here if needed.
            response = self.model.generate_content(prompt)
            
            # 6. Parse
            # AI Studio responses are accessed via .text
            return json.loads(response.text)

        except Exception as e:
            print(f"AI Studio Error: {e}")
            # Common errors: 429 (Rate Limit), 500 (Server Error)
            return None

# --- Usage Example ---

# 1. Get your API Key (Set this in your terminal or .env file)
# export GEMINI_API_KEY="AIzaSy..."
my_api_key = os.environ.get("GEMINI_API_KEY")

if not my_api_key:
    print("Please set your GEMINI_API_KEY environment variable.")
else:
    # 2. Initialize
    mutator = AIStudioCRNMutator(api_key=my_api_key)

    # 3. Dummy Data
    crn_data = """
    Species: [X1, X2]
    Rxns: X1 -> X2; [MAK(1.0)]
    """

    # 4. Run
    print("Querying Gemini 1.5 Flash (Free Tier)...")
    result = mutator.propose_mutation(crn_data, "Create a toggle switch")

    if result:
        print("\n--- Success ---")
        print("AI Reasoning:", result.get('reasoning'))
        print("New Reactions:", result.get('new_reactions'))

/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Vertex AI Error: 403 This API method requires billing to be enabled. Please enable billing on project #crn-evolution by visiting https://console.developers.google.com/billing/enable?project=crn-evolution then retry. If you enabled billing for this project recently, wait a few minutes for the action to propagate to our systems and retry. [reason: "BILLING_DISABLED"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "aiplatform.googleapis.com"
}
metadata {
  key: "containerInfo"
  value: "crn-evolution"
}
metadata {
  key: "consumer"
  value: "projects/crn-evolution"
}
, locale: "en-US"
message: "This API method requires billing to be enabled. Please enable billing on project #crn-evolution by visiting https://console.developers.google.com/billing/enable?project=crn-evolution then retry. If you enabled billing for this project recently, wait a few minutes for the action to propagate to our systems and retry."
, links {
  description: "Google developers console billing"
  url: 